In [ ]:
import time
import requests
import pandas as pd
from urllib.request import urlretrieve
from google.colab import userdata

## Keyword search

In [40]:
api_key = 'anwgY3hftPeAiVeZO2ne7QS06SMTp3VNZm8oAYsKCpRz2f5Y'
your_query = 'State Senator'  

# Keyword search
url = f'https://api.nytimes.com/svc/search/v2/articlesearch.json?q={your_query}&api-key={api_key}'

response = requests.get(url)
data = response.json()

In [42]:
docs = data.get("response", {}).get("docs", [])

rows = []
for d in docs:
    headline = (d.get("headline") or {}).get("main", "")

    # keyword > name == "Person" -> values
    person_vals = [
        k.get("value") for k in (d.get("keywords") or [])
        if k.get("name") == "Person" and k.get("value")
    ]

    mm = d.get("multimedia") or {}
    default_url = (mm.get("default") or {}).get("url", "")
    caption = mm.get("caption", "")

    rows.append({
        "headline": headline,
        "person_keywords": "; ".join(person_vals),
        "multimedia_default_url": default_url,
        "caption": caption,
        "news_desk": d.get("news_desk", ""),
        "section_name": d.get("section_name", ""),
        "subsection_name": d.get("subsection_name", ""),
        "source": d.get("source", ""),
        "web_url": d.get("web_url", ""),
    })

df = pd.DataFrame(rows)
df

,headline,person_keywords,multimedia_default_url,caption,news_desk,section_name,subsection_name,source,web_url
0,Colorado State Senator Is Killed in Car Crash,,https://static01.nyt.com/images/2025/11/27/mul...,Faith Winter in 2023. She was elected to the C...,Express,U.S.,,The New York Times,https://www.nytimes.com/2025/11/27/us/faith-wi...
1,‘Latinas for Trump’ Co-Founder Warns Immigrati...,"Miller, Stephen (1985- ); Trump, Donald J",https://static01.nyt.com/images/2026/01/23/us/...,Florida State Sen. Ileana Garcia sees herself ...,National,U.S.,,The New York Times,https://www.nytimes.com/2026/01/27/us/ileana-g...
2,Colorado State Senator at Fault in Car Crash T...,,https://static01.nyt.com/images/2025/12/20/mul...,A memorial service for state Senator Faith Win...,Express,U.S.,,The New York Times,https://www.nytimes.com/2025/12/20/us/colorado...
3,Minnesota State Senator Says Accused Gunman Vi...,"Boelter, Vance L; Hoffman, John A (1965- ); Ho...",https://static01.nyt.com/images/2025/06/16/mul...,"State Senator Ann Rest, right, at the State Ca...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/06/16/us/ann-rest...
4,"Florida Democratic Party Is ‘Dead,’ State Sena...","Pizzo, Jason WB",https://static01.nyt.com/images/2025/04/24/mul...,"Jason W.B. Pizzo, the highest-ranking Democrat...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/04/24/us/pizzo-fl...
5,"Dear Ohio State Senators: I’m a Student, Not a...",,https://static01.nyt.com/images/2023/03/25/opi...,Michelle Huang wrote about the issue of child ...,Learning,The Learning Network,,The New York Times,https://www.nytimes.com/2025/06/24/learning/de...
6,"Minnesota Lawmaker Convicted of Burglary, Leav...","Mitchell, Nicole (1974- )",https://static01.nyt.com/images/2025/07/18/mul...,"State Senator Nicole Mitchell, left, during op...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/07/18/us/minnesot...
7,Indiana Lawmakers Reject Trump’s New Political...,"Trump, Donald J",https://static01.nyt.com/images/2025/12/11/mul...,Legislators at the Indiana Statehouse on Thurs...,National,U.S.,,The New York Times,https://www.nytimes.com/2025/12/11/us/indiana-...
8,Amy Klobuchar Files Papers for Run to Succeed ...,"Klobuchar, Amy; Walz, Tim",https://static01.nyt.com/images/2026/01/05/mul...,Senator Amy Klobuchar of Minnesota is widely e...,Politics,U.S.,Politics,The New York Times,https://www.nytimes.com/2026/01/22/us/politics...
9,Why Texas Is the Wildest Story in U.S. Politic...,"Talarico, James; Crockett, Jasmine; Allred, Co...",https://static01.nyt.com/images/2026/02/02/mul...,"Taylor Rehmet, a Democrat, won an upset victor...",Politics,U.S.,Politics,The New York Times,https://www.nytimes.com/2026/02/02/us/politics...


In [43]:
df_long = df.assign(person_keywords=df["person_keywords"].str.split(r";\s*")).explode("person_keywords")
df_long["person_keywords"] = df_long["person_keywords"].fillna("")
df_long

,headline,person_keywords,multimedia_default_url,caption,news_desk,section_name,subsection_name,source,web_url
0,Colorado State Senator Is Killed in Car Crash,,https://static01.nyt.com/images/2025/11/27/mul...,Faith Winter in 2023. She was elected to the C...,Express,U.S.,,The New York Times,https://www.nytimes.com/2025/11/27/us/faith-wi...
1,‘Latinas for Trump’ Co-Founder Warns Immigrati...,"Miller, Stephen (1985- )",https://static01.nyt.com/images/2026/01/23/us/...,Florida State Sen. Ileana Garcia sees herself ...,National,U.S.,,The New York Times,https://www.nytimes.com/2026/01/27/us/ileana-g...
1,‘Latinas for Trump’ Co-Founder Warns Immigrati...,"Trump, Donald J",https://static01.nyt.com/images/2026/01/23/us/...,Florida State Sen. Ileana Garcia sees herself ...,National,U.S.,,The New York Times,https://www.nytimes.com/2026/01/27/us/ileana-g...
2,Colorado State Senator at Fault in Car Crash T...,,https://static01.nyt.com/images/2025/12/20/mul...,A memorial service for state Senator Faith Win...,Express,U.S.,,The New York Times,https://www.nytimes.com/2025/12/20/us/colorado...
3,Minnesota State Senator Says Accused Gunman Vi...,"Boelter, Vance L",https://static01.nyt.com/images/2025/06/16/mul...,"State Senator Ann Rest, right, at the State Ca...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/06/16/us/ann-rest...
3,Minnesota State Senator Says Accused Gunman Vi...,"Hoffman, John A (1965- )",https://static01.nyt.com/images/2025/06/16/mul...,"State Senator Ann Rest, right, at the State Ca...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/06/16/us/ann-rest...
3,Minnesota State Senator Says Accused Gunman Vi...,"Hortman, Melissa (1970-2025)",https://static01.nyt.com/images/2025/06/16/mul...,"State Senator Ann Rest, right, at the State Ca...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/06/16/us/ann-rest...
4,"Florida Democratic Party Is ‘Dead,’ State Sena...","Pizzo, Jason WB",https://static01.nyt.com/images/2025/04/24/mul...,"Jason W.B. Pizzo, the highest-ranking Democrat...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/04/24/us/pizzo-fl...
5,"Dear Ohio State Senators: I’m a Student, Not a...",,https://static01.nyt.com/images/2023/03/25/opi...,Michelle Huang wrote about the issue of child ...,Learning,The Learning Network,,The New York Times,https://www.nytimes.com/2025/06/24/learning/de...
6,"Minnesota Lawmaker Convicted of Burglary, Leav...","Mitchell, Nicole (1974- )",https://static01.nyt.com/images/2025/07/18/mul...,"State Senator Nicole Mitchell, left, during op...",National,U.S.,,The New York Times,https://www.nytimes.com/2025/07/18/us/minnesot...


## Archive

In [31]:
year = 2025
month = 12

url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key={api_key}"

response = requests.get(url)
data = response.json()

docs = data.get("response", {}).get("docs", [])

rows = []
for d in docs:
    headline = (d.get("headline") or {}).get("main", "")

    # keyword > name == "Person" -> values
    person_vals = [
        k.get("value") for k in (d.get("keywords") or [])
        if k.get("name") == "Person" and k.get("value")
    ]

    mm = d.get("multimedia") or {}
    default_url = (mm.get("default") or {}).get("url", "")
    caption = mm.get("caption", "")

    rows.append({
        "headline": headline,
        "person_keywords": "; ".join(person_vals),
        "multimedia_default_url": default_url,
        "caption": caption,
        "news_desk": d.get("news_desk", ""),
        "section_name": d.get("section_name", ""),
        "subsection_name": d.get("subsection_name", ""),
        "source": d.get("source", ""),
        "web_url": d.get("web_url", ""),
    })

df_dec = pd.DataFrame(rows)
df_dec

,headline,person_keywords,multimedia_default_url,caption,news_desk,section_name,subsection_name,source,web_url
0,,,,,,,,,
1,"‘It: Welcome to Derry’ Season 1, Episode 6 Rec...",,https://static01.nyt.com/images/2025/11/30/art...,It turns out Ingrid (Madeline Stowe) has a clo...,Culture,Arts,Television,The New York Times,https://www.nytimes.com/2025/11/30/arts/televi...
2,"Get Ready, America: Here Come China’s Food and...",,https://static01.nyt.com/images/2025/11/28/mul...,Staff members preparing drinks at Heytea in Ti...,Business,Business,,The New York Times,https://www.nytimes.com/2025/12/01/business/ch...
3,A Smartphone Before Age 12 Could Carry Health ...,,https://static01.nyt.com/images/2025/12/01/wel...,,Well,Well,Family,The New York Times,https://www.nytimes.com/2025/12/01/well/family...
4,Quote of the Day: Ukrainian Civilians Captured...,,,,Summary,Corrections,,The New York Times,https://www.nytimes.com/2025/12/01/pageoneplus...
...,...,...,...,...,...,...,...,...,...
3564,"In Chief Justice’s Annual Report, a History Le...","Roberts, John G Jr; Boasberg, James E; Trump, ...",https://static01.nyt.com/images/2025/12/31/mul...,Chief Justice John G. Roberts Jr. in May. By t...,Washington,U.S.,Politics,The New York Times,https://www.nytimes.com/2025/12/31/us/politics...
3565,"Mamdani Names Top Deputies for Child Care, Ope...","Hochul, Kathleen C; Mamdani, Zohran; Samuels, ...",https://static01.nyt.com/images/2025/12/31/mul...,"Emmy Liss, a veteran education leader, will be...",Metro,New York,,The New York Times,https://www.nytimes.com/2025/12/31/nyregion/ma...
3566,53 Fascinating Facts of 2025,,https://static01.nyt.com/images/2026/01/01/pag...,,Insider,Times Insider,,The New York Times,https://www.nytimes.com/2025/12/31/insider/202...
3567,"On His Last Day, Adams Drops From View but Sti...","Adams, Eric L",https://static01.nyt.com/images/2025/12/31/mul...,,Metro,New York,,The New York Times,https://www.nytimes.com/2025/12/31/nyregion/er...


In [32]:
df_all = pd.concat(
    [
        df_jan, df_feb, df_mar, df_apr,
        df_may, df_jun, df_jul, df_aug,
        df_sep, df_oct, df_nov, df_dec
    ],
    ignore_index=True
)

In [36]:
df_all.to_csv('df_all.csv', index=False)

In [ ]:
df_all.head(10)

In [33]:
df_long = df_all.assign(person_keywords=df_all["person_keywords"].str.split(r";\s*")).explode("person_keywords")
df_long["person_keywords"] = df_long["person_keywords"].fillna("")
df_long

,headline,person_keywords,multimedia_default_url,caption,news_desk,section_name,subsection_name,source,web_url
0,Dartmouth College Basketball Players Halt Effo...,"Trump, Donald J",https://static01.nyt.com/images/2024/12/31/mul...,The Dartmouth men’s basketball team playing ag...,National,U.S.,,The New York Times,https://www.nytimes.com/2024/12/31/us/dartmout...
1,‘He Saved Our Lives’: Former Hostages Recall C...,"Carter, Jimmy",https://static01.nyt.com/images/2024/12/31/us/...,"Former President Jimmy Carter, center, with so...",Washington,U.S.,Politics,The New York Times,https://www.nytimes.com/2024/12/31/us/politics...
1,‘He Saved Our Lives’: Former Hostages Recall C...,"Laingen, L Bruce",https://static01.nyt.com/images/2024/12/31/us/...,"Former President Jimmy Carter, center, with so...",Washington,U.S.,Politics,The New York Times,https://www.nytimes.com/2024/12/31/us/politics...
2,"Corrections: Jan. 1, 2025",,,,Corrections,Corrections,,The New York Times,https://www.nytimes.com/2024/12/31/pageoneplus...
3,A Certain Tankful,,https://static01.nyt.com/images/2024/12/30/cro...,Solving today’s crossword will leave you all a...,Games,Gameplay,,The New York Times,https://www.nytimes.com/2024/12/31/crosswords/...
...,...,...,...,...,...,...,...,...,...
47810,"Mamdani Names Top Deputies for Child Care, Ope...","Mamdani, Zohran",https://static01.nyt.com/images/2025/12/31/mul...,"Emmy Liss, a veteran education leader, will be...",Metro,New York,,The New York Times,https://www.nytimes.com/2025/12/31/nyregion/ma...
47810,"Mamdani Names Top Deputies for Child Care, Ope...","Samuels, Kamar",https://static01.nyt.com/images/2025/12/31/mul...,"Emmy Liss, a veteran education leader, will be...",Metro,New York,,The New York Times,https://www.nytimes.com/2025/12/31/nyregion/ma...
47811,53 Fascinating Facts of 2025,,https://static01.nyt.com/images/2026/01/01/pag...,,Insider,Times Insider,,The New York Times,https://www.nytimes.com/2025/12/31/insider/202...
47812,"On His Last Day, Adams Drops From View but Sti...","Adams, Eric L",https://static01.nyt.com/images/2025/12/31/mul...,,Metro,New York,,The New York Times,https://www.nytimes.com/2025/12/31/nyregion/er...


In [35]:
df_long.value_counts("person_keywords").head(30)

,count
person_keywords,
,19694
"Trump, Donald J",11965
"Musk, Elon",1061
"Biden, Joseph R Jr",701
"Mamdani, Zohran",701
"Putin, Vladimir V",694
"Zelensky, Volodymyr",511
"Netanyahu, Benjamin",497
"Adams, Eric L",480


In [ ]:
## High salience - national figures
# L: Joe Biden, Kamala Harris, Tim Walz, Johran Mamdani, Eric Adams, Andrew Cuomo, Gavin Newsom, Kathleen Hochul
# C: Donald Trump, Vance JD, Marco Rubio, Robert Kennedy Jr, Pete Hegseth, Marco Rubio, Charlie Kirk, Pamela Bondi, Bessent Scott, Mike Johnson
# Other: Elon Musk, Jefery Epstein, Charlie Kirk, Jimmy Kimmel, Stephen Colbert, Leo XIV

## Low sailence - state senate and house members


In [ ]:
df_long

# Keywords for each cell

In [ ]:
CELLS = [
    # (leaning, actor_structure, salience)
    ("liberal", "elite", "high"),
    ("conservative", "elite", "high"),
    ("liberal", "elite", "low"),
    ("conservative", "elite", "low"),
    ("liberal", "mass", "high"),
    ("conservative", "mass", "high"),
    ("liberal", "mass", "low"),
    ("conservative", "mass", "low"),
]

QUERY_BANK: Dict[Tuple[str, str, str], List[str]] = {
    ("liberal", "elite", "high"): [
        '("President" OR "Governor" OR "Senator") AND ("Democrat" OR "Democratic") AND ("speech" OR "press conference" OR "bill signing")',
        '("White House" OR "Capitol Hill") AND ("Democrat" OR "Democratic") AND ("press conference" OR "remarks")',
    ],
    ("conservative", "elite", "high"): [
        '("President" OR "Governor" OR "Senator") AND ("Republican") AND ("speech" OR "press conference" OR "bill signing")',
        '("Capitol Hill" OR "statehouse") AND ("Republican") AND ("press conference" OR "remarks")',
    ],
    ("liberal", "elite", "low"): [
        '("State Representative" OR "State Senator" OR "Attorney General" OR "Secretary of State") AND ("Democrat" OR "Democratic") AND ("committee hearing" OR "introduced bill" OR "policy meeting")',
        '("city council member" OR "county commissioner") AND ("Democrat" OR "Democratic") AND ("meeting" OR "hearing")',
    ],
    ("conservative", "elite", "low"): [
        '("State Representative" OR "State Senator" OR "Attorney General" OR "Secretary of State") AND ("Republican") AND ("committee hearing" OR "introduced bill" OR "policy meeting")',
        '("city council member" OR "county commissioner") AND ("Republican") AND ("meeting" OR "hearing")',
    ],
    ("liberal", "mass", "high"): [
        '("march" OR "rally" OR "demonstration") AND ("abortion rights" OR "gun control" OR "voting rights") AND ("thousands" OR "large crowd")',
        '("protest" OR "demonstration") AND ("climate" OR "immigration rights") AND ("thousands" OR "large crowd")',
    ],
    ("conservative", "mass", "high"): [
        '("march" OR "rally" OR "demonstration") AND ("anti-abortion" OR "gun rights" OR "election integrity") AND ("thousands" OR "large crowd")',
        '("protest" OR "rally") AND ("border security" OR "taxes") AND ("thousands" OR "large crowd")',
    ],
    ("liberal", "mass", "low"): [
        '("school board meeting" OR "city hall protest" OR "local rally") AND ("teachers" OR "public funding" OR "voting access")',
        '("community meeting" OR "town hall") AND ("public funding" OR "tenant" OR "housing") AND ("protest" OR "rally")',
    ],
    ("conservative", "mass", "low"): [
        '("school board meeting" OR "city hall protest" OR "local rally") AND ("parents" OR "tax protest" OR "election integrity")',
        '("community meeting" OR "town hall") AND ("parents" OR "taxes" OR "zoning") AND ("protest" OR "rally")',
    ],
}